https://github.com/ijstokes/tutorial-local-llm

---

# Local LLM

---

In [ ]:
# load in libraries
import json
from ollama import Client
from pathlib import Path
import pandas as pd

In [ ]:
# Select the condition
CONDITION = "AIG"

# Set the path to the processed passages
DATA_DIR = Path("..") / "pmc" / CONDITION / "processed"

# Load the passages
passages = pd.read_parquet(DATA_DIR / "passages.parquet")

# Check the data
print("Total passages:", len(passages))
display(passages.head(3))

In [ ]:
# select one passage (for testing)
passage = passages.iloc[0]["text"]
# Display the selected passage
print(passage)

In [ ]:
# Connect to the local Ollama server
client = Client(host="http://localhost:11434")

# select the local llm to use
MODEL = "qwen3:4b"

In [ ]:
# create instructions for extracting diagnostic criteria
PROMPT = f"""
Extract diagnostic criteria from the following medical passage.

Only extract information explicitly stated in the passage.
Do not invent diagnostic criteria or thresholds.

Return JSON containing:
- criteria: a list of diagnostic criteria
- evidence: the exact supporting text for each criterion

If no diagnostic criteria are found, return an empty list.

Passage:
{passage}
"""

In [ ]:
prompt = PROMPT

In [ ]:
# Create a function to send prompts to the local LLM
def run_llm(prompt, model=MODEL):

    # Send the prompt to the local LLM
    response = chat(
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        format="json",
        options={"temperature": 0}
    )

    # Convert the JSON response into a dictionary
    result = json.loads(response.message.content)

    return result

In [ ]:
# Extract diagnostic criteria from the passage
result = run_llm(PROMPT)

In [ ]:
# Display the JSON output
print(json.dumps(result, indent=2))